In [18]:
import json
import pandas as pd
from pathlib import Path

# -----------------------------
# Config
# -----------------------------
drug_list = [
    "Abemaciclib", "Alpelisib", "Anastrozole", "Atezolizumab", "Bevacizumab",
    "Capivasertib", "Dostarlimab", "Elacestrant", "Entrectinib", "Everolimus",
    "Exemestane", "Fulvestrant", "Lapatinib", "Larotrectinib", "Letrozole",
    "Margetuximab", "Neratinib", "Olaparib", "Paclitaxel", "Palbociclib",
    "Pembrolizumab", "Pertuzumab", "Ribociclib", "Sacituzumab govitecan",
    "Selpercatinib", "Talazoparib", "Tamoxifen", "Toremifene", "Trastuzumab",
    "Trastuzumab deruxtecan", "Trastuzumab emtansine", "Tucatinib"
]

EXCEL_PATH = Path("Deep_search_results/deepsearch_3_ADMINISTRATION_regulation_effect_rationale_citation_tab.xlsx")
INPUT_JSON_DIR = Path("drug_deepsearch_json_final")   # e.g., drug_deepsearch_json_final/Paclitaxel.json
OUTPUT_JSON_DIR = Path("drug_json_final")             # e.g., drug_json_final/Paclitaxel_final_2.json
OUTPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

PATHWAY_ROOT = "pathway_sets_annotations"

# Column names (must match your Excel)
COL_DRUG = "Drug"
COL_PATHWAY = "Pathway"
COL_ADMIN = "Administration"
COL_REG = "Regulation"
COL_EFFECT = "Effect"
COL_RATIONALE = "Rationale"
COL_REFS = "Ref(s) (PMIDs, IDs, titles)"

# -----------------------------
# Helpers
# -----------------------------
def normalize_key(s: str) -> str:
    """
    Normalize pathway keys to avoid false mismatches due to:
      - trailing/leading whitespace
      - multiple spaces/tabs/newlines
      - non-breaking spaces
      - zero-width chars
    """
    if s is None:
        return ""
    s = str(s)
    s = s.replace("\u00A0", " ").replace("\u200b", "")  # NBSP, zero-width space
    s = " ".join(s.split())  # collapse whitespace + strip
    return s

def normalize_effect(effect: str) -> str:
    if effect is None:
        raise ValueError("Effect is None")

    e = str(effect).strip()
    if e.lower() in {"sensitive", "sensitivity"}:
        return "Sensitive"
    if e.lower() in {"resistance", "resistant"}:
        return "Resistance"
    return e

def normalize_admin(admin: str) -> str:
    if admin is None:
        raise ValueError("Administration is None")
    a = str(admin).strip()
    if a.lower() == "before":
        return "Before"
    if a.lower() == "after":
        return "After"
    return a

def normalize_reg(reg: str) -> str:
    if reg is None:
        raise ValueError("Regulation is None")
    r = str(reg).strip()
    if r.lower() == "up":
        return "Up"
    if r.lower() == "down":
        return "Down"
    return r

# -----------------------------
# Load Excel once
# -----------------------------
df = pd.read_excel(EXCEL_PATH)

# Basic column validation (optional but useful)
required_cols = {COL_DRUG, COL_PATHWAY, COL_ADMIN, COL_REG, COL_EFFECT, COL_RATIONALE, COL_REFS}
missing = required_cols - set(df.columns)
if missing:
    raise KeyError(f"Excel is missing required columns: {missing}")

# -----------------------------
# Main loop
# -----------------------------
for drug_name in drug_list:
    input_json_path = INPUT_JSON_DIR / f"{drug_name}.json"
    if not input_json_path.exists():
        raise FileNotFoundError(f"Missing JSON for drug '{drug_name}': {input_json_path}")

    print(f"\n==============================")
    print(f"[START] Processing drug: {drug_name}")
    print(f"       JSON: {input_json_path}")
    print(f"==============================")

    # Load per-drug JSON
    with open(input_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Ensure pathway root exists
    if PATHWAY_ROOT not in data:
        data[PATHWAY_ROOT] = {}

    # Build a lookup map: normalized_json_key -> original_json_key
    # If duplicates occur after normalization, that's a real data issue.
    json_key_map = {}
    for original_key in data[PATHWAY_ROOT].keys():
        norm = normalize_key(original_key)
        if norm in json_key_map and json_key_map[norm] != original_key:
            raise ValueError(
                f"[ERROR] Duplicate JSON pathway keys after normalization for drug '{drug_name}':\n"
                f"  normalized: {repr(norm)}\n"
                f"  key1: {repr(json_key_map[norm])}\n"
                f"  key2: {repr(original_key)}\n"
                f"Fix JSON keys so they normalize uniquely."
            )
        json_key_map[norm] = original_key

    # Filter Excel rows for this drug
    df_drug = df[df[COL_DRUG] == drug_name].copy()
    if df_drug.empty:
        print(f"[WARN] No rows found in Excel for drug '{drug_name}'. Saving JSON unchanged.")
        out_path = OUTPUT_JSON_DIR / f"{drug_name}_final_2.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        print(f"[SAVED] {drug_name} -> {out_path}")
        continue

    ADMIN_KEY = f"Administration_of_{drug_name.lower().replace(' ', '_')}"

    # Get pathways listed in Excel for this drug (normalized for consistent iteration)
    df_drug[COL_PATHWAY] = df_drug[COL_PATHWAY].astype(str)
    pathways_in_excel_norm = (
        df_drug[COL_PATHWAY]
        .dropna()
        .map(normalize_key)
        .unique()
        .tolist()
    )

    for pathway_norm in pathways_in_excel_norm:
        # ---- REQUIRED: raise error if pathway not present in JSON (after normalization) ----
        if pathway_norm not in json_key_map:
            # show a small hint of what's available
            sample_keys = list(data[PATHWAY_ROOT].keys())[:15]
            raise KeyError(
                f"[ERROR] Pathway '{pathway_norm}' (drug='{drug_name}') not found in JSON under "
                f"data['{PATHWAY_ROOT}'] (after normalization).\n"
                f"Example JSON keys (first 15): {sample_keys}\n"
                f"Tip: check spelling differences, suffixes, or missing pathway mapping."
            )

        # Use the original JSON key (preserves JSON as-is)
        pathway_json_key = json_key_map[pathway_norm]

        # Subset rows for this pathway (normalize each row pathway for match)
        df_pw = df_drug[df_drug[COL_PATHWAY].map(normalize_key) == pathway_norm]

        pathway_obj = data[PATHWAY_ROOT].setdefault(pathway_json_key, {})
        admin_obj = pathway_obj.setdefault(ADMIN_KEY, {})

        for _, row in df_pw.iterrows():
            administration = normalize_admin(row[COL_ADMIN])       # Before/After
            regulation = normalize_reg(row[COL_REG])               # Up/Down
            effect = normalize_effect(row[COL_EFFECT])             # Sensitive/Resistance
            rationale = row[COL_RATIONALE]
            refs = row[COL_REFS]

            # Ensure "Before"/"After" level
            admin_section = admin_obj.setdefault(administration, {})

            # Ensure "Pathway_regulation" level
            reg_root = admin_section.setdefault("Pathway_regulation", {})

            # Ensure "Up"/"Down" level
            reg_section = reg_root.setdefault(regulation, {})

            # Ensure effect list
            effect_list = reg_section.setdefault(effect, [])

            # Append [rationale, refs]
            effect_list.append([rationale, refs])

        print(f"[DONE] {drug_name} :: pathway completed -> {pathway_json_key} (rows: {len(df_pw)})")

    # Save updated JSON
    out_path = OUTPUT_JSON_DIR / f"{drug_name}_final.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"[SAVED] {drug_name} -> {out_path}")



[START] Processing drug: Abemaciclib
       JSON: drug_deepsearch_json_final\Abemaciclib.json
[DONE] Abemaciclib :: pathway completed -> HALLMARK_E2F_TARGETS (rows: 8)
[DONE] Abemaciclib :: pathway completed -> HALLMARK_ESTROGEN_RESPONSE_EARLY (rows: 8)
[DONE] Abemaciclib :: pathway completed -> REACTOME_CYCLIN_D_ASSOCIATED_EVENTS_IN_G1 (rows: 8)
[DONE] Abemaciclib :: pathway completed -> REACTOME_CYCLIN_E_ASSOCIATED_EVENTS_DURING_G1_S_TRANSITION (rows: 8)
[DONE] Abemaciclib :: pathway completed -> REACTOME_ABERRANT_REGULATION_OF_MITOTIC_G1_S_TRANSITION_IN_CANCER_DUE_TO_RB1_DEFECTS (rows: 8)
[DONE] Abemaciclib :: pathway completed -> HALLMARK_PI3K_AKT_MTOR_SIGNALING (rows: 8)
[DONE] Abemaciclib :: pathway completed -> REACTOME_SIGNALING_BY_FGFR1 (rows: 8)
[DONE] Abemaciclib :: pathway completed -> KEGG_MEDICUS_REFERENCE_P16_CELL_CYCLE_G1_S (rows: 8)
[DONE] Abemaciclib :: pathway completed -> KEGG_MEDICUS_REFERENCE_PDL_PD1_SHP_PI3K_SIGNALING_PATHWAY (rows: 8)
[DONE] Abemaciclib :: path

In [1]:
from msigdb import MsigDB

# Example input list
pathway_names = ["HALLMARK_APOPTOSIS", "HALLMARK_HYPOXIA"]

# Load MSigDB (defaults to the latest release and species="human")
db = MsigDB()

# Fetch gene sets for each pathway in the list
gene_sets = {}

for name in pathway_names:
    # query by gene set name exactly
    result = db.get_gene_set(name)
    if result is not None:
        gene_sets[name] = result['genes']
    else:
        print(f"Warning: pathway '{name}' not found in MSigDB.")

# Print results
for name, genes in gene_sets.items():
    print(f"\n{name} ({len(genes)} genes):")
    print(genes)


ModuleNotFoundError: No module named 'msigdb'

In [3]:
!pip install gsea_api

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for gsea_api: filename=gsea_api-0.3.4-py3-none-any.whl size=16259 sha256=5c2b4166935a79da767407333b3c79618debecba7c38187044c757bac59ad0c4
  Stored in directory: c:\users\jeet\appdata\local\pip\cache\wheels\b3\9a\65\ca96bfc4d03cc97b9f52cb40036ea19390e49080af14f69f35
Successfully built gsea_api


In [4]:
from gsea_api.molecular_signatures_db import MolecularSignaturesDatabase

In [5]:
msigdb = MolecularSignaturesDatabase('msigdb', version=7.1)
msigdb.gene_sets

ValueError: Could not find MSigDB: msigdb does not exist

In [6]:
import pandas as pd

In [8]:
xyz  = pd.read_csv("MSigDB/msigdb_v2025.1.Hs_files_to_download_locally/msigdb_v2025.1.Hs_files_to_download_locally/msigdb_v2025.1.Hs_GMTs/h.all.v2025.1.Hs.symbols.gmt", sep="\t", header="None")
xyz

ValueError: header must be integer or list of integers

In [11]:
import gseapy as gp

gmt_file = "MSigDB\msigdb_v2025.1.Hs_files_to_download_locally\msigdb_v2025.1.Hs_files_to_download_locally\msigdb_v2025.1.Hs_GMTs\h.all.v2025.1.Hs.symbols.gmt"

genesets = gp.parser.gsea_cls_parser(gmt_file)

print(genesets["HALLMARK_ADIPOGENESIS"][:10])

<>:3: SyntaxWarning: invalid escape sequence '\m'
<>:3: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Jeet\AppData\Local\Temp\ipykernel_13092\1795375188.py:3: SyntaxWarning: invalid escape sequence '\m'
  gmt_file = "MSigDB\msigdb_v2025.1.Hs_files_to_download_locally\msigdb_v2025.1.Hs_files_to_download_locally\msigdb_v2025.1.Hs_GMTs\h.all.v2025.1.Hs.symbols.gmt"
C:\Users\Jeet\AppData\Local\Temp\ipykernel_13092\1795375188.py:3: SyntaxWarning: invalid escape sequence '\m'
  gmt_file = "MSigDB\msigdb_v2025.1.Hs_files_to_download_locally\msigdb_v2025.1.Hs_files_to_download_locally\msigdb_v2025.1.Hs_GMTs\h.all.v2025.1.Hs.symbols.gmt"


Exception: Input groups have to be 2!

In [ ]:
def read_gmt(filepath):
    genesets = {}

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            name = parts[0]                   # e.g., HALLMARK_ADIPOGENESIS
            url = parts[1]                    # second element (ignored if not needed)
            genes = parts[2:]                 # rest are genes
            genesets[name] = genes

    return genesets


# Example usage:
gmt_path ="MSigDB\msigdb_v2025.1.Hs_files_to_download_locally\msigdb_v2025.1.Hs_files_to_download_locally\msigdb_v2025.1.Hs_GMTs\msigdb.v2025.1.Hs.symbols.gmt"
genesets = read_gmt(gmt_path)

# print(genesets["HALLMARK_ADIPOGENESIS"][:10])   # first 10 genes


<>:16: SyntaxWarning: invalid escape sequence '\m'
<>:16: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Jeet\AppData\Local\Temp\ipykernel_13092\280302694.py:16: SyntaxWarning: invalid escape sequence '\m'
  gmt_path ="MSigDB\msigdb_v2025.1.Hs_files_to_download_locally\msigdb_v2025.1.Hs_files_to_download_locally\msigdb_v2025.1.Hs_GMTs\msigdb.v2025.1.Hs.symbols.gmt"


['ABCA1', 'ABCB8', 'ACAA2', 'ACADL', 'ACADM', 'ACADS', 'ACLY', 'ACO2', 'ACOX1', 'ADCY6']


In [15]:
genesets

{'MT': ['MT-ATP6',
  'MT-ATP8',
  'MT-CO1',
  'MT-CO2',
  'MT-CO3',
  'MT-CYB',
  'MT-ND1',
  'MT-ND2',
  'MT-ND3',
  'MT-ND4',
  'MT-ND4L',
  'MT-ND5',
  'MT-ND6',
  'MT-RNR1',
  'MT-RNR2',
  'MT-TA',
  'MT-TC',
  'MT-TD',
  'MT-TE',
  'MT-TF',
  'MT-TG',
  'MT-TH',
  'MT-TI',
  'MT-TK',
  'MT-TL1',
  'MT-TL2',
  'MT-TM',
  'MT-TN',
  'MT-TP',
  'MT-TQ',
  'MT-TR',
  'MT-TS1',
  'MT-TS2',
  'MT-TT',
  'MT-TV',
  'MT-TW',
  'MT-TY'],
 'chr10p11': ['ABCD1P2',
  'ACTR3BP5',
  'AK3P5',
  'ANKRD30A',
  'ARHGAP12',
  'ARL6IP1P2',
  'ATP6V1G1P4',
  'C1DP1',
  'CCDC7',
  'CCND3P1',
  'CCNY',
  'CCNY-AS1',
  'CCNYL4',
  'CHEK2P5',
  'CICP9',
  'CKS1BP2',
  'CREM',
  'CUL2',
  'DDX10P1',
  'DNM1P17',
  'EEF1A1P39',
  'EIF3LP3',
  'ELOBP4',
  'ENSG00000223834',
  'ENSG00000274458',
  'ENSG00000285630',
  'ENSG00000289616',
  'ENSG00000299623',
  'ENSG00000308646',
  'EPC1',
  'EPC1-AS1',
  'EPC1-AS2',
  'FXYD6P2',
  'FZD8',
  'GJD4',
  'GOLGA2P6',
  'HMGB1P7',
  'HNRNPA1P32',
  'HSD17B7P2',
  'I

In [20]:
print(genesets["WP_ESTROGEN_SIGNALING"][:10])   # first 10 genes


['AKT1', 'BCL2', 'BRAF', 'CHUK', 'CREB1', 'ELK1', 'ESR1', 'FOS', 'GNAS', 'GNB1']
